# k-Nearest Neighbours

KNN is the simplest classifier to state: to label a new point, find its **k**
nearest training points and take a majority vote. There's no "training" beyond
storing the data — all the work happens at prediction time. The one real choice is
**k**, and it's a textbook bias/variance knob:

- small **k** → flexible, jagged boundary that chases noise (high variance);
- large **k** → smooth boundary that may wash out real structure (high bias).

We show that visually on a 2-D slice of `smartcore`'s breast-cancer data, then
measure how k affects accuracy on the full 30-feature dataset. This is the first
of three [classification](knn-classification.ipynb) models we'll compare.

In [ ]:
:dep smartcore = { version = "0.3", features = ["datasets"] }
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::linalg::basic::arrays::{Array, Array2};
use smartcore::dataset::breast_cancer;
use smartcore::model_selection::train_test_split;
use smartcore::metrics::accuracy;
use smartcore::neighbors::knn_classifier::{KNNClassifier, KNNClassifierParameters};
use plotters::prelude::*;

// Load full data; also build a 2-feature standardized view (features 0 & 1) for
// the decision-region plots. Persisted values are plain Vecs / explicitly-typed
// matrices so evcxr keeps them across cells.
let (feat2, y, n): (Vec<f32>, Vec<i32>, usize) = {
    let ds = breast_cancer::load_dataset();
    let (n, p) = (ds.num_samples, ds.num_features);
    let y: Vec<i32> = ds.target.iter().map(|&v| v as i32).collect();
    // means/stds of features 0 and 1
    let mut mean = [0f64; 2];
    for i in 0..n { for j in 0..2 { mean[j] += ds.data[i * p + j] as f64; } }
    for j in 0..2 { mean[j] /= n as f64; }
    let mut sd = [0f64; 2];
    for i in 0..n { for j in 0..2 { let d = ds.data[i * p + j] as f64 - mean[j]; sd[j] += d * d; } }
    for j in 0..2 { sd[j] = (sd[j] / n as f64).sqrt(); }
    let mut feat2 = Vec::with_capacity(n * 2);
    for i in 0..n { for j in 0..2 { feat2.push(((ds.data[i * p + j] as f64 - mean[j]) / sd[j]) as f32); } }
    (feat2, y, n)
};
// full 30-feature matrix (rebuilt from the dataset for the accuracy section)
let x_full: DenseMatrix<f32> = {
    let ds = breast_cancer::load_dataset();
    DenseMatrix::new(ds.num_samples, ds.num_features, ds.data.clone(), false)
};
let x2: DenseMatrix<f32> = DenseMatrix::new(n, 2, feat2.clone(), false);
println!("{} samples; 2-D view = standardized features 0 & 1", n);

## Decision regions as k grows

Fit KNN on the 2-D view, then classify every point on a fine grid and colour the
background by predicted class (blue = benign, orange = malignant), with the
training points on top. Watch the boundary go from **jagged (k=3)** to **smooth
(k=51)**:

In [ ]:
let ks = [3usize, 15, 51];
// grid extent from the standardized data
let (mut x0, mut x1, mut y0, mut y1) = (f32::MAX, f32::MIN, f32::MAX, f32::MIN);
for i in 0..n {
    x0 = x0.min(feat2[i * 2]); x1 = x1.max(feat2[i * 2]);
    y0 = y0.min(feat2[i * 2 + 1]); y1 = y1.max(feat2[i * 2 + 1]);
}
let steps = 60usize;
let grid: DenseMatrix<f32> = {
    let mut g = Vec::with_capacity(steps * steps * 2);
    for gi in 0..steps {
        for gj in 0..steps {
            g.push(x0 + (x1 - x0) * gi as f32 / (steps - 1) as f32);
            g.push(y0 + (y1 - y0) * gj as f32 / (steps - 1) as f32);
        }
    }
    DenseMatrix::new(steps * steps, 2, g, false)
};

evcxr_figure((900, 320), |root| {
    root.fill(&WHITE)?;
    let panels = root.split_evenly((1, 3));
    for (panel, &k) in panels.iter().zip(ks.iter()) {
        let model = KNNClassifier::fit(&x2, &y, KNNClassifierParameters::default().with_k(k)).unwrap();
        let gpred = model.predict(&grid).unwrap();
        let mut chart = ChartBuilder::on(panel)
            .caption(format!("k = {}", k), ("sans-serif", 15))
            .margin(5).x_label_area_size(24).y_label_area_size(28)
            .build_cartesian_2d(x0..x1, y0..y1)?;
        chart.configure_mesh().disable_mesh().draw()?;
        // background decision regions
        let cw = (x1 - x0) / (steps - 1) as f32;
        let ch = (y1 - y0) / (steps - 1) as f32;
        chart.draw_series((0..steps * steps).map(|idx| {
            let gi = (idx / steps) as f32;
            let gj = (idx % steps) as f32;
            let cx = x0 + (x1 - x0) * gi / (steps - 1) as f32;
            let cy = y0 + (y1 - y0) * gj / (steps - 1) as f32;
            let color = if gpred[idx] == 1 { RGBColor(255, 224, 189) } else { RGBColor(200, 220, 255) };
            Rectangle::new([(cx - cw / 2.0, cy - ch / 2.0), (cx + cw / 2.0, cy + ch / 2.0)], color.filled())
        }))?;
        // training points (subsample for clarity)
        chart.draw_series((0..n).step_by(2).map(|i| {
            let color = if y[i] == 1 { RGBColor(220, 120, 20) } else { RGBColor(30, 90, 200) };
            Circle::new((feat2[i * 2], feat2[i * 2 + 1]), 2, color.filled())
        }))?;
    }
    Ok(())
})

The k=3 panel carves tight islands around stray points; k=51 gives a clean,
almost-linear frontier. Neither extreme is automatically best — that's what the
accuracy sweep below is for.

## How k affects accuracy (full 30 features)

Now the honest measurement: on the full dataset with a held-out test split, sweep
k and plot train vs. test accuracy — the same [validation-curve](../01d-evaluation/learning-curves.ipynb)
diagnostic from the evaluation chapter, applied to KNN's k.

In [ ]:
let ks_full: Vec<usize> = vec![3, 5, 9, 15, 25, 41, 61];
let (tr, te): (Vec<(f64, f64)>, Vec<(f64, f64)>) = {
    let (xtr, xte, ytr, yte) = train_test_split(&x_full, &y, 0.3, true, Some(42));
    let mut tr = vec![];
    let mut te = vec![];
    for &k in &ks_full {
        let model = KNNClassifier::fit(&xtr, &ytr, KNNClassifierParameters::default().with_k(k)).unwrap();
        tr.push((k as f64, accuracy(&ytr, &model.predict(&xtr).unwrap())));
        te.push((k as f64, accuracy(&yte, &model.predict(&xte).unwrap())));
    }
    (tr, te)
};
for (i, &k) in ks_full.iter().enumerate() {
    println!("k={:>2}:  train acc = {:.3},  test acc = {:.3}", k, tr[i].1, te[i].1);
}

evcxr_figure((580, 400), |root| {
    root.fill(&WHITE)?;
    let kmax = *ks_full.last().unwrap() as f64;
    let mut chart = ChartBuilder::on(&root)
        .caption("KNN accuracy vs k", ("sans-serif", 16))
        .margin(10).x_label_area_size(36).y_label_area_size(44)
        .build_cartesian_2d(3f64..kmax, 0.85f64..1.005f64)?;
    chart.configure_mesh().x_desc("k").y_desc("accuracy").draw()?;
    chart.draw_series(LineSeries::new(tr.clone(), BLUE.stroke_width(2)))?
        .label("train").legend(|(x, y)| PathElement::new(vec![(x, y), (x + 18, y)], BLUE));
    chart.draw_series(LineSeries::new(te.clone(), RED.stroke_width(2)))?
        .label("test").legend(|(x, y)| PathElement::new(vec![(x, y), (x + 18, y)], RED));
    chart.draw_series(te.iter().map(|p| Circle::new(*p, 3, RED.filled())))?;
    chart.configure_series_labels().position(SeriesLabelPosition::LowerLeft)
        .background_style(WHITE.mix(0.85)).border_style(BLACK).draw()?;
    Ok(())
})

At small k, train accuracy sits well above test — the model memorises noise
(overfitting). As k grows the two converge, and the best test accuracy lands at a
moderate k. (`smartcore`'s KNN requires k > 1, so we start at k=3.)

## A note on distance

KNN's other choice is the **distance metric**. `smartcore` defaults to Euclidean
(`with_distance` also offers Manhattan, Minkowski, Mahalanobis). Euclidean is the
sensible default here because we **standardized** the features first — without
that, a large-range feature would dominate the distance and the neighbours would
be decided by one column. Scaling matters for every distance-based method (the
same point the [ETL chapter](../01c-etl/data-preparation.ipynb) made about
clustering).

Next: [Naive Bayes](naive-bayes.ipynb) — a probabilistic classifier that, unlike
KNN, is nearly free at prediction time.